In [ ]:
# Cell 0 — FIX-11: Label Validation Gate (MANDATORY — run before training)
import json, pandas as pd
from pathlib import Path
from collections import Counter

REPO_ROOT = Path("..").resolve()
train_path = REPO_ROOT / "data" / "lora_training" / "signal_train.jsonl"

train_df = pd.read_json(str(train_path), lines=True)
print(f"Total training examples: {len(train_df)}")

def get_pattern(row):
    try:
        return json.loads(row["output"])["autonomic_pattern"]
    except Exception:
        return "PARSE_ERROR"

patterns = train_df["output"].apply(get_pattern)
print("\nLabel distribution:")
print(patterns.value_counts())

# Imbalance warning: flag any class < 15% of data
pcts = patterns.value_counts(normalize=True)
for p, pct in pcts.items():
    if pct < 0.15:
        print(f"WARNING: '{p}' underrepresented ({pct:.1%}) — model may underperform on this class")

print("\n5 random samples for manual review:")
for _, row in train_df.sample(5, random_state=42).iterrows():
    out = json.loads(row["output"])
    print(f"  pattern={out['autonomic_pattern']} | input={row['input'][:80]}...")
    print(f"  reasoning: {out['physiological_reasoning'][:80]}...")

# Hard gate — requires manual confirmation before training cell can run
confirmed = input("\nHave you reviewed the labels and class balance? Type 'yes' to proceed: ")
assert confirmed.strip().lower() == "yes", "Training aborted — labels not confirmed by user"
print("Labels confirmed. Proceed to Cell 1.")

In [ ]:
# Cell 1 — Imports and config
import sys, torch
from pathlib import Path

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_NAME  = "microsoft/Phi-3-mini-4k-instruct"
ADAPTER_DIR = REPO_ROOT / "models" / "exports" / "signal_specialist_lora"

# Apple Silicon: use MPS if available, else CPU.
# bitsandbytes (4-bit quant) is excluded — not reliably supported on M2 MPS.
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE  = torch.float16
print(f"Device: {DEVICE} | dtype: {DTYPE}")
print(f"Adapter will be saved to: {ADAPTER_DIR}")

In [ ]:
# Cell 2 — Load training data
import json, pandas as pd
from datasets import Dataset

train_path = REPO_ROOT / "data" / "lora_training" / "signal_train.jsonl"
train_df = pd.read_json(str(train_path), lines=True)

def make_prompt(row):
    return (
        f"### Instruction:\n{row['instruction']}\n\n"
        f"### Input:\n{row['input']}\n\n"
        f"### Output:\n{row['output']}"
    )

train_df["text"] = train_df.apply(make_prompt, axis=1)
dataset = Dataset.from_pandas(train_df[["text"]])
print(f"Training examples: {len(dataset)}")
print("\nSample prompt (first 300 chars):")
print(dataset[0]["text"][:300])

In [ ]:
# Cell 3 — Load Phi-3-mini + apply LoRA
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model ({DTYPE}, device={DEVICE}) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    device_map=DEVICE,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA applied.")

In [ ]:
# Cell 4 — QLoRA training
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=(DEVICE != "cpu"),         # float16 on MPS; pure float32 on CPU
    bf16=False,                     # bfloat16 not supported on MPS
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    report_to="none",               # no wandb/tensorboard
    dataset_text_field="text",
    max_seq_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training ...")
trainer.train()

# Save LoRA adapter weights only (not full model — saves disk space)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"\nAdapter saved to: {ADAPTER_DIR}")

In [ ]:
# Cell 5 — Inference smoke-test (offline, no Groq)
# Tests that the trained adapter produces parseable SignalAssessment JSON.
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from eval.scenarios import SCENARIOS

# Load adapter fresh (confirms save worked)
print("Loading saved adapter for inference test ...")
tok2 = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
if tok2.pad_token is None:
    tok2.pad_token = tok2.eos_token

base2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True, device_map=DEVICE
)
lora_model = PeftModel.from_pretrained(base2, str(ADAPTER_DIR))
lora_model.eval()

instruction = "Classify the neonatal HRV autonomic pattern from these z-score deviations from the infant's personal baseline. Do NOT recommend clinical actions."

for s in SCENARIOS[:5]:
    z_parts = ", ".join(f"{k} z={v:+.2f}" for k, v in list(s.z_scores.items())[:4])
    inp = f"{z_parts}. Risk score {s.risk_score:.2f}. Bradycardia events: {s.n_brady}."
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Output:\n"

    inputs = tok2(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = lora_model.generate(
            **inputs, max_new_tokens=200, do_sample=False,
            pad_token_id=tok2.pad_token_id
        )
    decoded = tok2.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Try parsing as JSON
    try:
        j_start = decoded.find("{")
        j_end   = decoded.rfind("}") + 1
        parsed = json.loads(decoded[j_start:j_end]) if j_start != -1 and j_end > 0 else {}
        pattern = parsed.get("autonomic_pattern", "PARSE_FAILED")
    except Exception:
        pattern = "PARSE_FAILED"

    expected_level = s.expected
    print(f"  {s.patient_id:25s} expected={expected_level} | lora_pattern={pattern}")

print("\nInference smoke-test complete. Patterns above should be non-empty and JSON-parseable.")
print("If pattern=PARSE_FAILED: model needs more training epochs or prompt tuning.")